
# Cyclistic Bike Share — Process Phase

This notebook covers the **Process** phase of the Cyclistic case study.

## Goal
Prepare a clean, analysis-ready dataset from the 12 monthly Cyclistic / Divvy trip files by:

- loading all raw CSV files from `01_Raw_Data`
- checking schema consistency and basic data quality
- standardizing column names and data types
- appending the monthly files into one master table
- creating derived columns for later analysis
- removing invalid records
- validating the cleaned output
- exporting cleaned files to `02_Processed_Data`

## Outputs
- `cyclistic_cleaned_trips.csv`
- `cyclistic_sql_ready_trips.csv`
- `schema_audit.csv`
- `null_report.csv`
- `validation_summary.csv`
- `cleaning_log.csv`
- `processing_summary.json`


In [1]:

# Core imports
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import pyarrow

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.options.display.float_format = "{:,.2f}".format


In [ ]:
%pip install pyarrow --quiet


## 1. Configuration


In [2]:

# Resolve the project root robustly
if Path.cwd().name == "03_Scripts":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "01_Raw_Data"
PROCESSED_DIR = PROJECT_ROOT / "02_Processed_Data"
SCRIPTS_DIR = PROJECT_ROOT / "03_Scripts"
SQL_DIR = PROJECT_ROOT / "04_SQL_Queries"
VIZ_DIR = PROJECT_ROOT / "05_Visualizations"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Optional filter for extreme ride durations
APPLY_MAX_DURATION_FILTER = False
MAX_REASONABLE_RIDE_MINUTES = 24 * 60  # 24 hours

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR exists:", RAW_DIR.exists())
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())
print("Files in RAW_DIR:", len(list(RAW_DIR.glob("*.csv"))))


PROJECT_ROOT: C:\Users\John Lloyd Legaspi\Documents\Projects\Cyclistic_Bike_Share
RAW_DIR exists: True
PROCESSED_DIR exists: True
Files in RAW_DIR: 12



## 2. Load raw files

In [35]:

csv_files = sorted(RAW_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {RAW_DIR}. Put your monthly Cyclistic trip files there first."
    )

file_inventory = pd.DataFrame(
    {
        "file_name": [f.name for f in csv_files],
        "file_size_mb": [round(f.stat().st_size / (1024 * 1024), 2) for f in csv_files],
    }
)

display(file_inventory)


,file_name,file_size_mb
0,202503-divvy-tripdata.csv,58.08
1,202504-divvy-tripdata.csv,72.14
2,202505-divvy-tripdata.csv,96.85
3,202506-divvy-tripdata.csv,129.66
4,202507-divvy-tripdata.csv,145.28
5,202508-divvy-tripdata.csv,150.25
6,202509-divvy-tripdata.csv,136.05
7,202510-divvy-tripdata.csv,122.41
8,202511-divvy-tripdata.csv,67.82
9,202512-divvy-tripdata.csv,26.99



## 3. Inspect schemas before merging

The goal here is to confirm:
- column names
- row counts
- whether all files share the same structure
- which columns may need standardization


In [4]:
audit_results = []

for file_path in csv_files:
    # Read the full file for an accurate audit
    df = pd.read_csv(file_path)
    
    # 1. Basic Stats & Missing Values
    row_count = len(df)
    missing_values = df.isnull().sum().to_dict()
    
    # 2. Duplicate Check
    duplicate_ids = df['ride_id'].duplicated().sum()
    
    # 3. Unique Value Inspection (Looking for "Unusual" entries)
    unique_member_types = df['member_casual'].unique().tolist()
    unique_bike_types = df['rideable_type'].unique().tolist()
    
    # 4. Timestamp & Duration Check
    # Attempt to convert to datetime to see if they are 'parsable'
    started_at = pd.to_datetime(df['started_at'], errors='coerce')
    ended_at = pd.to_datetime(df['ended_at'], errors='coerce')
    failed_timestamps = started_at.isna().sum() + ended_at.isna().sum()
    
    # Calculate ride length to find "Impossible" durations (e.g., < 0 or > 24hrs)
    # Reference: 
    ride_len = (ended_at - started_at).dt.total_seconds()
    impossible_durations = ((ride_len <= 0) | (ride_len > 86400)).sum()

    audit_results.append({
        "file_name": file_path.name,
        "total_rows": row_count,
        "missing_stations": missing_values.get('start_station_name', 0),
        "duplicate_ride_ids": duplicate_ids,
        "failed_timestamp_parse": failed_timestamps,
        "impossible_durations": impossible_durations,
        "member_types": unique_member_types,
        "bike_types": unique_bike_types,
        "column_count": len(df.columns),
        "columns": ", ".join(df.columns)
    })

# Create the Audit DataFrame
full_audit = pd.DataFrame(audit_results)

# Display a summary table for quick inspection
display(full_audit[[
    "file_name", "total_rows", "duplicate_ride_ids", 
    "impossible_durations", "failed_timestamp_parse"
]])

# Save the audit for your "Documentation of Cleaning" deliverable [cite: 101]
full_audit.to_csv(PROCESSED_DIR / "comprehensive_audit_report.csv", index=False)

,file_name,total_rows,duplicate_ride_ids,impossible_durations,failed_timestamp_parse
0,202503-divvy-tripdata.csv,298155,0,251,0
1,202504-divvy-tripdata.csv,371341,0,300,0
2,202505-divvy-tripdata.csv,502456,0,563,0
3,202506-divvy-tripdata.csv,678904,0,983,0
4,202507-divvy-tripdata.csv,763432,0,978,0
5,202508-divvy-tripdata.csv,790177,0,702,0
6,202509-divvy-tripdata.csv,714759,0,618,0
7,202510-divvy-tripdata.csv,646039,0,599,0
8,202511-divvy-tripdata.csv,356628,0,366,0
9,202512-divvy-tripdata.csv,140534,0,125,0



## 4. Standardize the schema


In [6]:
# 1. Define the exact order and names for the final merged file 
canonical_columns = [
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id", "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"
]

def standardize_columns(df: pd.DataFrame) -> tuple:
    """
    Standardizes a single month's dataframe and generates a health report.
    Returns a tuple of (DataFrame, health_metrics_dict).
    """
    df = df.copy()
    report = {}

    # 1. Header Sanity Check
    # Ensure all required columns exist before we try to reorder them [cite: 138]
    df.columns = [c.strip().lower() for c in df.columns]
    missing_cols = set(canonical_columns) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing critical columns: {missing_cols}")

    # 2. Timestamp Conversion & Audit 
    # We log if conversion results in nulls (NaT)
    df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
    df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce')
    report['timestamp_errors'] = df['started_at'].isna().sum() + df['ended_at'].isna().sum()

    # 3. Categorical Value Sanity Check [cite: 96]
    # We log unique values to ensure no 'subscriber' or 'customer' leftovers
    report['user_types'] = df['member_casual'].unique().tolist()
    report['bike_types'] = df['rideable_type'].unique().tolist()

    # 4. Text Field Cleanup [cite: 98]
    string_cols = ["ride_id", "rideable_type", "start_station_name", 
                   "start_station_id", "end_station_name", "end_station_id", "member_casual"]
    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # 5. Final Schema Alignment [cite: 138]
    df = df[canonical_columns]
    
    return df, report


## 5. Load, standardize, and append all files


In [7]:
import pandas as pd

frames = []
ingestion_log = []

# Loop through and process each file 
for file_path in csv_files:
    # 1. Load raw data [cite: 103]
    temp = pd.read_csv(file_path, low_memory=False)
    raw_rows = len(temp)
    
    # 2. Unpack the DataFrame and the Health Report from your function
    # This checks for errors and ensures data integrity [cite: 91, 96]
    temp, health_report = standardize_columns(temp)
    
    # 3. Add metadata to track the source file
    temp["source_file"] = file_path.name
    
    frames.append(temp)

    # 4. Merge basic log data with the detailed health report [cite: 99]
    # We use **health_report to "flatten" the dictionary into the log
    log_entry = {
        "file_name": file_path.name,
        "raw_rows": raw_rows,
        "standardized_rows": len(temp),
    }
    log_entry.update(health_report) # Adds timestamp_errors, user_types, etc.
    
    ingestion_log.append(log_entry)
    print(f"Verified and appended: {file_path.name}")

# 5. Create the final full-year merged DataFrame 
trips = pd.concat(frames, ignore_index=True)

# 6. Display the Detailed Ingestion & Health Log
ingestion_log_df = pd.DataFrame(ingestion_log)
display(ingestion_log_df)

print(f"Total Rows Ingested: {len(trips):,}")
trips.head()

Verified and appended: 202503-divvy-tripdata.csv
Verified and appended: 202504-divvy-tripdata.csv
Verified and appended: 202505-divvy-tripdata.csv
Verified and appended: 202506-divvy-tripdata.csv
Verified and appended: 202507-divvy-tripdata.csv
Verified and appended: 202508-divvy-tripdata.csv
Verified and appended: 202509-divvy-tripdata.csv
Verified and appended: 202510-divvy-tripdata.csv
Verified and appended: 202511-divvy-tripdata.csv
Verified and appended: 202512-divvy-tripdata.csv
Verified and appended: 202601-divvy-tripdata.csv
Verified and appended: 202602-divvy-tripdata.csv


,file_name,raw_rows,standardized_rows,timestamp_errors,user_types,bike_types
0,202503-divvy-tripdata.csv,298155,298155,0,"[member, casual]","[electric_bike, classic_bike]"
1,202504-divvy-tripdata.csv,371341,371341,0,"[member, casual]","[classic_bike, electric_bike]"
2,202505-divvy-tripdata.csv,502456,502456,0,"[member, casual]","[classic_bike, electric_bike]"
3,202506-divvy-tripdata.csv,678904,678904,0,"[member, casual]","[electric_bike, classic_bike]"
4,202507-divvy-tripdata.csv,763432,763432,0,"[member, casual]","[classic_bike, electric_bike]"
5,202508-divvy-tripdata.csv,790177,790177,0,"[casual, member]","[electric_bike, classic_bike]"
6,202509-divvy-tripdata.csv,714759,714759,0,"[member, casual]","[electric_bike, classic_bike]"
7,202510-divvy-tripdata.csv,646039,646039,0,"[member, casual]","[classic_bike, electric_bike]"
8,202511-divvy-tripdata.csv,356628,356628,0,"[casual, member]","[electric_bike, classic_bike]"
9,202512-divvy-tripdata.csv,140534,140534,0,"[member, casual]","[electric_bike, classic_bike]"


Total Rows Ingested: 5,601,662


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file
0,16CBE9844D401954,electric_bike,2025-03-18 08:39:20.065,2025-03-18 08:51:37.633,<NA>,<NA>,Canal St & Jackson Blvd,13138,41.91,-87.67,41.88,-87.64,member,202503-divvy-tripdata.csv
1,1CB408029E2B5F74,electric_bike,2025-03-24 16:04:22.239,2025-03-24 16:27:41.347,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.91,-87.71,member,202503-divvy-tripdata.csv
2,7B6A76CD0F204D08,electric_bike,2025-03-10 16:06:19.708,2025-03-10 16:29:17.457,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.91,-87.71,member,202503-divvy-tripdata.csv
3,4F7084E3D75CDE31,electric_bike,2025-03-21 14:28:14.579,2025-03-21 14:35:06.160,<NA>,<NA>,Canal St & Jackson Blvd,13138,41.87,-87.63,41.88,-87.64,member,202503-divvy-tripdata.csv
4,E419A570A5A0475B,electric_bike,2025-03-14 17:54:14.484,2025-03-14 18:17:53.254,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.89,-87.67,41.91,-87.71,casual,202503-divvy-tripdata.csv



## 6. Initial quality checks

This is a high-level profile before cleaning:
- duplicates
- nulls
- distinct rider categories
- date parsing readiness


In [8]:

initial_row_count = len(trips)
initial_duplicate_ride_ids = trips["ride_id"].duplicated().sum()

null_report_initial = (
    trips.isna()
    .sum()
    .rename("null_count")
    .to_frame()
    .assign(null_pct=lambda x: (x["null_count"] / len(trips) * 100).round(2))
    .sort_values("null_count", ascending=False)
)

print("Initial row count:", f"{initial_row_count:,}")
print("Duplicate ride_id count:", f"{initial_duplicate_ride_ids:,}")
print("Distinct member_casual values:", trips["member_casual"].dropna().unique())
print("Distinct rideable_type values:", trips["rideable_type"].dropna().unique())

display(null_report_initial)


Initial row count: 5,601,662
Duplicate ride_id count: 0
Distinct member_casual values: <ArrowStringArray>
['member', 'casual']
Length: 2, dtype: string
Distinct rideable_type values: <ArrowStringArray>
['electric_bike', 'classic_bike']
Length: 2, dtype: string


,null_count,null_pct
end_station_id,1254439,22.39
end_station_name,1254439,22.39
start_station_name,1192515,21.29
start_station_id,1192515,21.29
end_lat,5742,0.10
end_lng,5742,0.10
started_at,0,0.00
ended_at,0,0.00
ride_id,0,0.00
rideable_type,0,0.00


In [9]:
# Profile the rides missing end coordinates
missing_end_coords = trips[trips['end_lat'].isna()]

print("--- Profile of Rides Missing End Coordinates ---")
print(missing_end_coords['member_casual'].value_counts())
print(missing_end_coords['rideable_type'].value_counts())

--- Profile of Rides Missing End Coordinates ---
member_casual
casual    4735
member    1007
Name: count, dtype: int64[pyarrow]
rideable_type
classic_bike    5742
Name: count, dtype: int64[pyarrow]



## 7. Convert data types and create derived columns

These fields support later analysis:
- `ride_length`
- `ride_date`
- `ride_month`
- `ride_day`
- `month_name`
- `day_of_week`
- `day_name`
- `ride_hour`
- `is_weekend`
- `is_round_trip`


In [10]:
# Parse timestamps
trips["started_at"] = pd.to_datetime(trips["started_at"], errors="coerce")
trips["ended_at"] = pd.to_datetime(trips["ended_at"], errors="coerce")

# Convert numeric columns where possible
for col in ["start_lat", "start_lng", "end_lat", "end_lng"]:
    trips[col] = pd.to_numeric(trips[col], errors="coerce")

# Derived columns
# Formula: Ride Length = (Ended - Started) / 60
trips["ride_length"] = (trips["ended_at"] - trips["started_at"]).dt.total_seconds() / 60

trips["ride_date"] = trips["started_at"].dt.date
trips["ride_month"] = trips["started_at"].dt.month
trips["month_name"] = trips["started_at"].dt.month_name()
trips["ride_day"] = trips["started_at"].dt.day # Day of the month (1-31)

# Case Study Requirement: 1=Sunday, 7=Saturday 
trips["day_of_week"] = (trips["started_at"].dt.dayofweek + 1) % 7 + 1 
trips["day_name"] = trips["started_at"].dt.day_name()
trips["ride_hour"] = trips["started_at"].dt.hour

# 3. Categorical Flags for Analysis
# If 1=Sunday and 7=Saturday, then weekend is [1, 7]
trips["is_weekend"] = trips["day_of_week"].isin([1, 7])

# Detect "Round Trips" (Started and ended at the same station)
trips["is_round_trip"] = (
    trips["start_station_name"].fillna("Unknown") == trips["end_station_name"].fillna("Unknown")
)

trips.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file,ride_length,ride_date,ride_month,month_name,ride_day,day_of_week,day_name,ride_hour,is_weekend,is_round_trip
0,16CBE9844D401954,electric_bike,2025-03-18 08:39:20.065,2025-03-18 08:51:37.633,<NA>,<NA>,Canal St & Jackson Blvd,13138,41.91,-87.67,41.88,-87.64,member,202503-divvy-tripdata.csv,12.29,2025-03-18,3,March,18,3,Tuesday,8,False,False
1,1CB408029E2B5F74,electric_bike,2025-03-24 16:04:22.239,2025-03-24 16:27:41.347,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.91,-87.71,member,202503-divvy-tripdata.csv,23.32,2025-03-24,3,March,24,2,Monday,16,False,False
2,7B6A76CD0F204D08,electric_bike,2025-03-10 16:06:19.708,2025-03-10 16:29:17.457,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.91,-87.71,member,202503-divvy-tripdata.csv,22.96,2025-03-10,3,March,10,2,Monday,16,False,False
3,4F7084E3D75CDE31,electric_bike,2025-03-21 14:28:14.579,2025-03-21 14:35:06.160,<NA>,<NA>,Canal St & Jackson Blvd,13138,41.87,-87.63,41.88,-87.64,member,202503-divvy-tripdata.csv,6.86,2025-03-21,3,March,21,6,Friday,14,False,False
4,E419A570A5A0475B,electric_bike,2025-03-14 17:54:14.484,2025-03-14 18:17:53.254,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.89,-87.67,41.91,-87.71,casual,202503-divvy-tripdata.csv,23.65,2025-03-14,3,March,14,6,Friday,17,False,False


In [ ]:
# Save the 'Pre-Cleaned' Master File
# This represents the 'Raw Merged' state with derived columns 
trips.to_parquet(PROCESSED_DIR / "trips_raw_merged.parquet", index=False)
print(f"Checkpoint saved: {PROCESSED_DIR / 'trips_raw_merged.parquet'}")

In [11]:
parquet_file = PROCESSED_DIR / "trips_raw_merged.parquet"

# Load the file back into memory
trips = pd.read_parquet(parquet_file, engine='pyarrow')

print(f"Successfully reloaded {len(trips):,} rows.")
trips.head()

Successfully reloaded 5,601,662 rows.


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file,ride_length,ride_date,ride_month,month_name,ride_day,day_of_week,day_name,ride_hour,is_weekend,is_round_trip
0,16CBE9844D401954,electric_bike,2025-03-18 08:39:20.065,2025-03-18 08:51:37.633,<NA>,<NA>,Canal St & Jackson Blvd,13138,41.91,-87.67,41.88,-87.64,member,202503-divvy-tripdata.csv,12.29,2025-03-18,3,March,18,3,Tuesday,8,False,False
1,1CB408029E2B5F74,electric_bike,2025-03-24 16:04:22.239,2025-03-24 16:27:41.347,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.91,-87.71,member,202503-divvy-tripdata.csv,23.32,2025-03-24,3,March,24,2,Monday,16,False,False
2,7B6A76CD0F204D08,electric_bike,2025-03-10 16:06:19.708,2025-03-10 16:29:17.457,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.91,-87.71,member,202503-divvy-tripdata.csv,22.96,2025-03-10,3,March,10,2,Monday,16,False,False
3,4F7084E3D75CDE31,electric_bike,2025-03-21 14:28:14.579,2025-03-21 14:35:06.160,<NA>,<NA>,Canal St & Jackson Blvd,13138,41.87,-87.63,41.88,-87.64,member,202503-divvy-tripdata.csv,6.86,2025-03-21,3,March,21,6,Friday,14,False,False
4,E419A570A5A0475B,electric_bike,2025-03-14 17:54:14.484,2025-03-14 18:17:53.254,<NA>,<NA>,Albany Ave & Bloomingdale Ave,15655,41.89,-87.67,41.91,-87.71,casual,202503-divvy-tripdata.csv,23.65,2025-03-14,3,March,14,6,Friday,17,False,False



## 8. Clean invalid records

Cleaning rules used here:
1. remove exact duplicate rows
2. remove duplicate `ride_id` values, keeping the first record
3. remove rows missing key fields: `ride_id`, `started_at`, `ended_at`, `member_casual`
4. remove rides with non-positive duration
5. optionally remove rides above a maximum duration threshold


In [12]:
# 1. Flag the two types of anomalies
trips['is_missing_coords'] = trips['end_lat'].isna()
trips['is_impossible_dur'] = (trips['ride_length'] <= 0) | (trips['ride_length'] > 1440)

# 2. Create a Cross-tabulation
overlap_analysis = pd.crosstab(
    trips['is_missing_coords'], 
    trips['is_impossible_dur'], 
    rownames=['Missing End Coords'], 
    colnames=['Impossible Duration']
)

print("--- Overlap Analysis: Coordinates vs. Duration ---")
display(overlap_analysis)

# 3. Percentage of Missing Coords that are also Impossible Durations
overlap_pct = (overlap_analysis.loc[True, True] / overlap_analysis.loc[True].sum() * 100).round(2)
print(f"\n{overlap_pct}% of rides with missing coordinates also have impossible durations.")

--- Overlap Analysis: Coordinates vs. Duration ---


Impossible Duration,False,True
Missing End Coords,,
False,5595683,237
True,186,5556



96.76% of rides with missing coordinates also have impossible durations.


In [14]:
# Define the filters for impossible durations
too_short = trips[trips['ride_length'] <= 0]
too_long = trips[trips['ride_length'] > 1440]

# Print the Summary Report for your documentation [cite: 99]
print("--- Impossible Duration Audit ---")
print(f"Rides <= 0 minutes:  {len(too_short):,}")
print(f"Rides > 24 hours:    {len(too_long):,}")
print(f"Total Anomalies:     {len(too_short) + len(too_long):,}")

# Deep Dive: Breakdown by User Type
# This helps answer: "Why would casual riders buy memberships?" [cite: 42]
anomaly_by_user = trips[
    (trips['ride_length'] <= 0) | (trips['ride_length'] > 1440)
].groupby('member_casual').size()

print("\n--- Anomalies by User Type ---")
print(anomaly_by_user)

--- Impossible Duration Audit ---
Rides <= 0 minutes:  29
Rides > 24 hours:    5,764
Total Anomalies:     5,793

--- Anomalies by User Type ---
member_casual
casual    4764
member    1029
dtype: int64


In [15]:
# 1. Categorize specific duration issues
trips['dur_category'] = 'Normal'
trips.loc[trips['ride_length'] <= 0, 'dur_category'] = 'Short (<=0m)'
trips.loc[trips['ride_length'] > 1440, 'dur_category'] = 'Long (>24h)'

# 2. Flag missing coordinates
trips['missing_coords'] = trips['end_lat'].isna()

# 3. Create a detailed cross-tabulation to see the "fall in"
detailed_overlap = pd.crosstab(
    trips['dur_category'], 
    trips['missing_coords'],
    rownames=['Duration Category'],
    colnames=['Missing End Coords'],
    margins=True # Adds the 'All' total row/column
)

print("--- Detailed Anomaly Breakdown ---")
display(detailed_overlap)

# 4. Proactive Investigation: Which user type is "losing" bikes?
# This addresses why casual riders might be different from members [cite: 41, 42]
missing_and_long = trips[(trips['dur_category'] == 'Long (>24h)') & (trips['missing_coords'] == True)]
user_leakage = missing_and_long.groupby('member_casual').size()

print("\n--- Long Rides (>24h) with Missing Coordinates by User Type ---")
print(user_leakage)

--- Detailed Anomaly Breakdown ---


Missing End Coords,False,True,All
Duration Category,,,
Long (>24h),208,5556,5764
Normal,5595683,186,5595869
Short (<=0m),29,0,29
All,5595920,5742,5601662



--- Long Rides (>24h) with Missing Coordinates by User Type ---
member_casual
casual    4608
member     948
dtype: int64


In [16]:
# 1. Define thresholds for cleaning
MIN_RIDE_MINUTES = 1    # Removes false starts and mechanical issues
MAX_RIDE_MINUTES = 240  # 4 hours - removes outliers/forgotten returns
APPLY_MAX_DURATION_FILTER = True
key_fields = ["ride_id", "started_at", "ended_at", "member_casual"]

# 2. Calculate Discrepancy Counts
discrepancy_report = {
    "Exact Duplicate Rows": trips.duplicated().sum(),
    "Duplicate Ride IDs": trips.duplicated(subset=["ride_id"]).sum(),
    "Missing Key Fields": trips[key_fields].isna().any(axis=1).sum(),
    "Short Durations (<= 1 min)": (trips["ride_length"] < MIN_RIDE_MINUTES).sum(),
    "Long Durations (> 4h)": (trips["ride_length"] > MAX_RIDE_MINUTES).sum(),
    "Normal but Missing Coords": trips[(trips["ride_length"] >= MIN_RIDE_MINUTES) & (trips["ride_length"] <= MAX_RIDE_MINUTES) & (trips["end_lat"].isna())].shape[0]
}

# 3. Display the Report
print("--- Pre-Cleaning Discrepancy Report ---")
for issue, count in discrepancy_report.items():
    print(f"{issue}: {count:,}")

# Check for "Round Trips" with 0 duration (often maintenance tests)
round_trip_zero = trips[(trips["is_round_trip"]) & (trips["ride_length"] <= 0)].shape[0]
print(f"Potential Maintenance Tests (Round Trip + 0 min): {round_trip_zero:,}")

--- Pre-Cleaning Discrepancy Report ---
Exact Duplicate Rows: 0
Duplicate Ride IDs: 0
Missing Key Fields: 0
Short Durations (<= 1 min): 149,691
Long Durations (> 4h): 12,692
Normal but Missing Coords: 91
Potential Maintenance Tests (Round Trip + 0 min): 1


In [17]:
# Check how many rides are under 60 seconds
ultra_short_rides = trips[trips['ride_length'] < 1].shape[0]
rides_under_2_minutes = trips[trips['ride_length'] < 2].shape[0]
ultra_long_rides = trips[trips['ride_length'] > 1200].shape[0] # Rides over 20 hours

print(f"Rides under 1 minute: {ultra_short_rides:,}")
print(f"Rides under 2 minutes: {rides_under_2_minutes:,}")
print(f"Rides over 20 hours:  {ultra_long_rides:,}")

Rides under 1 minute: 149,691
Rides under 2 minutes: 265,298
Rides over 20 hours:  6,422


In [18]:
# Let's see how many people actually ride more than 3 hours (180 mins)
over_3_hours = trips[trips['ride_length'] > 180].shape[0]
pct_over_3 = (over_3_hours / len(trips) * 100)

over_4_hours = trips[trips['ride_length'] > 240].shape[0]
pct_over_4 = (over_4_hours / len(trips) * 100)

over_5_hours = trips[trips['ride_length'] > 300].shape[0]
pct_over_5 = (over_5_hours / len(trips) * 100)

over_8_hours = trips[trips['ride_length'] > 480].shape[0]
pct_over_8 = (over_8_hours / len(trips) * 100)

over_12_hours = trips[trips['ride_length'] > 720].shape[0]
pct_over_12 = (over_12_hours / len(trips) * 100)

print(f"Rides over 3 hours: {over_3_hours:,}")
print(f"Percentage of total data: {pct_over_3:.2f}%")
print(f"Rides over 4 hours: {over_4_hours:,}")
print(f"Percentage of total data: {pct_over_4:.2f}%")
print(f"Rides over 5 hours: {over_5_hours:,}")
print(f"Percentage of total data: {pct_over_5:.2f}%")
print(f"Rides over 8 hours: {over_8_hours:,}")
print(f"Percentage of total data: {pct_over_8:.2f}%")
print(f"Rides over 12 hours: {over_12_hours:,}")
print(f"Percentage of total data: {pct_over_12:.2f}%")


Rides over 3 hours: 16,713
Percentage of total data: 0.30%
Rides over 4 hours: 12,692
Percentage of total data: 0.23%
Rides over 5 hours: 11,022
Percentage of total data: 0.20%
Rides over 8 hours: 9,032
Percentage of total data: 0.16%
Rides over 12 hours: 8,138
Percentage of total data: 0.15%


In [19]:
# --- Informational Diagnostic: February 2025 ---
# Filter for February 2025 using the started_at timestamp
feb_2025_diagnostics = trips[
    (trips['started_at'] >= '2025-02-01') & 
    (trips['started_at'] <= '2025-02-28 23:59:59')
].copy()

# Add a duration column in seconds for granular detail
feb_2025_diagnostics['duration_seconds'] = (
    feb_2025_diagnostics['ended_at'] - feb_2025_diagnostics['started_at']
).dt.total_seconds()

print(f"--- Diagnostic Report: February 2025 ---")
print(f"Total out-of-scope records found: {len(feb_2025_diagnostics)}")
print(f"User Breakdown:\n{feb_2025_diagnostics['member_casual'].value_counts()}")
display(feb_2025_diagnostics.sort_values('started_at'))

--- Diagnostic Report: February 2025 ---
Total out-of-scope records found: 36
User Breakdown:
member_casual
member    25
casual    11
Name: count, dtype: int64[pyarrow]


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file,ride_length,ride_date,ride_month,month_name,ride_day,day_of_week,day_name,ride_hour,is_weekend,is_round_trip,is_missing_coords,is_impossible_dur,dur_category,missing_coords,duration_seconds
174055,5FBF1BFE4C72756F,classic_bike,2025-02-28 08:06:36.009,2025-03-01 09:06:25.127,Orleans St & Hubbard St,636,<NA>,<NA>,41.89,-87.64,NaN,NaN,member,202503-divvy-tripdata.csv,"1,499.82",2025-02-28,2,February,28,6,Friday,8,False,False,True,True,Long (>24h),True,"89,989.12"
198630,FDA49BB617F644B4,classic_bike,2025-02-28 12:29:55.028,2025-03-01 13:29:33.874,Sheridan Rd & Montrose Ave,TA1307000107,<NA>,<NA>,41.96,-87.65,NaN,NaN,casual,202503-divvy-tripdata.csv,"1,499.65",2025-02-28,2,February,28,6,Friday,12,False,False,True,True,Long (>24h),True,"89,978.85"
170597,8B20E2B0D8603957,classic_bike,2025-02-28 13:53:38.438,2025-03-01 14:53:31.545,DuSable Lake Shore Dr & Monroe St,13300,<NA>,<NA>,41.88,-87.62,NaN,NaN,casual,202503-divvy-tripdata.csv,"1,499.89",2025-02-28,2,February,28,6,Friday,13,False,False,True,True,Long (>24h),True,"89,993.11"
201115,B7FC6F4C1F021136,classic_bike,2025-02-28 13:58:06.709,2025-03-01 14:58:00.064,Broadway & Granville Ave,15571,<NA>,<NA>,41.99,-87.66,NaN,NaN,member,202503-divvy-tripdata.csv,"1,499.89",2025-02-28,2,February,28,6,Friday,13,False,False,True,True,Long (>24h),True,"89,993.35"
193693,F837FF70199A73B3,classic_bike,2025-02-28 14:02:54.583,2025-03-01 15:02:34.239,Phillips Ave & 83rd St,582,<NA>,<NA>,41.74,-87.57,NaN,NaN,casual,202503-divvy-tripdata.csv,"1,499.66",2025-02-28,2,February,28,6,Friday,14,False,False,True,True,Long (>24h),True,"89,979.66"
194345,AABE0F307027EE82,classic_bike,2025-02-28 14:11:52.177,2025-03-01 15:11:31.261,Wilton Ave & Diversey Pkwy,TA1306000014,<NA>,<NA>,41.93,-87.65,NaN,NaN,member,202503-divvy-tripdata.csv,"1,499.65",2025-02-28,2,February,28,6,Friday,14,False,False,True,True,Long (>24h),True,"89,979.08"
54310,3367A3B7A5C4A6F5,classic_bike,2025-02-28 15:52:42.648,2025-03-01 16:52:36.803,Kilbourn Ave & Irving Park Rd,590,<NA>,<NA>,41.95,-87.74,NaN,NaN,member,202503-divvy-tripdata.csv,"1,499.90",2025-02-28,2,February,28,6,Friday,15,False,False,True,True,Long (>24h),True,"89,994.15"
60822,CFA71F7010931BD2,classic_bike,2025-02-28 16:35:46.914,2025-03-01 17:35:42.046,Central Park Ave & North Ave,15652,<NA>,<NA>,41.91,-87.72,NaN,NaN,member,202503-divvy-tripdata.csv,"1,499.92",2025-02-28,2,February,28,6,Friday,16,False,False,True,True,Long (>24h),True,"89,995.13"
159189,12B14EB8299BA751,classic_bike,2025-02-28 17:39:22.633,2025-03-01 18:39:18.902,Western Ave & Fillmore St,644,<NA>,<NA>,41.87,-87.69,NaN,NaN,casual,202503-divvy-tripdata.csv,"1,499.94",2025-02-28,2,February,28,6,Friday,17,False,False,True,True,Long (>24h),True,"89,996.27"
211888,2C8C21C83E2E7B07,classic_bike,2025-02-28 19:20:43.552,2025-03-01 12:03:12.050,Sheffield Ave & Webster Ave,TA1309000033,Larrabee St & Menomonee St,TA1306000007,41.92,-87.65,41.91,-87.64,casual,202503-divvy-tripdata.csv,"1,002.47",2025-02-28,2,February,28,6,Friday,19,False,False,False,False,Normal,False,"60,148.50"


In [20]:
cleaning_log = []

def log_step(step_name: str, rows_before: int, rows_after: int) -> None:
    cleaning_log.append(
        {
            "step": step_name,
            "rows_before": rows_before,
            "rows_after": rows_after,
            "rows_removed": rows_before - rows_after,
        }
    )

# Step 1: Remove records prior to the official study start (March 1, 2025)
STUDY_START_DATE = "2025-03-01"
rows_before = len(trips)
trips = trips[trips["started_at"] >= STUDY_START_DATE].copy()
rows_after = len(trips)
log_step(f"Removed records prior to study start ({STUDY_START_DATE})", rows_before, rows_after)

# Step 2: remove exact duplicate rows
rows_before = len(trips)
trips = trips.drop_duplicates()
rows_after = len(trips)
log_step("Removed exact duplicate rows", rows_before, rows_after)

# Step 3: remove duplicate ride_id values, keeping first
rows_before = len(trips)
trips = trips.drop_duplicates(subset=["ride_id"], keep="first")
rows_after = len(trips)
log_step("Removed duplicate ride_id values", rows_before, rows_after)

# Step 4: remove rows missing key fields
key_fields = ["ride_id", "started_at", "ended_at", "member_casual"]
rows_before = len(trips)
trips = trips.dropna(subset=key_fields)
rows_after = len(trips)
log_step("Removed rows missing key fields", rows_before, rows_after)

# Step 5: Refined Minimum Duration Filter (1 Minute)
rows_before = len(trips)
trips = trips[trips["ride_length"] >= MIN_RIDE_MINUTES].copy()
log_step(f"Removed rides shorter than {MIN_RIDE_MINUTES} minute(s)", rows_before, len(trips))

# Step 6: Refined Maximum Duration Filter (4 Hours)
rows_before = len(trips)
trips = trips[trips["ride_length"] <= MAX_RIDE_MINUTES].copy()
log_step(f"Removed rides longer than {MAX_RIDE_MINUTES} minutes (4 hrs)", rows_before, len(trips))

# Step 7: remove rides missing spatial end data (optional but recommended for mapping)
rows_before = len(trips)
trips = trips.dropna(subset=["end_lat", "end_lng"])
rows_after = len(trips)
log_step("Removed rides missing end coordinates", rows_before, rows_after)

# Final cleanup for key text columns
for col in ["rideable_type", "start_station_name", "start_station_id",
            "end_station_name", "end_station_id", "member_casual"]:
    trips[col] = trips[col].astype("string").str.strip()

cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)
print("Final cleaned rows:", f"{len(trips):,}")

,step,rows_before,rows_after,rows_removed
0,Removed records prior to study start (2025-03-01),5601662,5601626,36
1,Removed exact duplicate rows,5601626,5601626,0
2,Removed duplicate ride_id values,5601626,5601626,0
3,Removed rows missing key fields,5601626,5601626,0
4,Removed rides shorter than 1 minute(s),5601626,5451935,149691
5,Removed rides longer than 240 minutes (4 hrs),5451935,5439253,12682
6,Removed rides missing end coordinates,5439253,5439162,91


Final cleaned rows: 5,439,162



## 9. Validate the cleaned dataset

These checks confirm that the dataset is ready for analysis.


In [21]:
validation_summary = {
    "final_row_count": int(len(trips)),
    "final_column_count": int(len(trips.columns)),
    "duplicate_ride_ids_remaining": int(trips["ride_id"].duplicated().sum()),
    "null_started_at": int(trips["started_at"].isna().sum()),
    "null_ended_at": int(trips["ended_at"].isna().sum()),
    "null_member_casual": int(trips["member_casual"].isna().sum()),
    
    # Coordinate Check: These should be 0 after your Step 6
    "null_end_lat": int(trips["end_lat"].isna().sum()),
    "null_end_lng": int(trips["end_lng"].isna().sum()),
    
    # Range Checks: Ensuring filtering worked
    "min_ride_length": float(trips["ride_length"].min()),
    "max_ride_length": float(trips["ride_length"].max()),

    # Spatial Sanity Check: Ensuring coordinates are within Chicago area
    "min_lat": float(trips["start_lat"].min()),
    "max_lat": float(trips["start_lat"].max()),
    "min_lng": float(trips["start_lng"].min()),
    "max_lng": float(trips["start_lng"].max()),
    
    # Temporal Check
    "date_min": str(trips["started_at"].min()),
    "date_max": str(trips["started_at"].max()),
    
    # Category Check
    "member_categories": sorted([str(x) for x in trips["member_casual"].dropna().unique()]),
    "rideable_types": sorted([str(x) for x in trips["rideable_type"].dropna().unique()]),
    
}

# Convert to DataFrame for a clean display
validation_summary_df = pd.DataFrame(
    validation_summary.items(), columns=["metric", "value"]
)

# Full Null Audit
null_report_final = (
    trips.isna()
    .sum()
    .rename("null_count")
    .to_frame()
    .assign(null_pct=lambda x: (x["null_count"] / len(trips) * 100).round(2))
    .sort_values("null_count", ascending=False)
)

print("--- Final Post-Cleaning Validation ---")
display(validation_summary_df)
print("\n--- Full Null Report ---")
display(null_report_final)

--- Final Post-Cleaning Validation ---


,metric,value
0,final_row_count,5439162
1,final_column_count,28
2,duplicate_ride_ids_remaining,0
3,null_started_at,0
4,null_ended_at,0
5,null_member_casual,0
6,null_end_lat,0
7,null_end_lng,0
8,min_ride_length,1.00
9,max_ride_length,239.97



--- Full Null Report ---


,null_count,null_pct
end_station_name,1135231,20.87
end_station_id,1135231,20.87
start_station_name,1101029,20.24
start_station_id,1101029,20.24
ended_at,0,0.00
started_at,0,0.00
rideable_type,0,0.00
ride_id,0,0.00
start_lat,0,0.00
start_lng,0,0.00



## 10. Quick sanity tables for analysis handoff

These are not the final analysis, but they help confirm the cleaned data behaves as expected.


In [22]:

rider_mix = (
    trips.groupby("member_casual", dropna=False)
    .agg(
        trips=("ride_id", "count"),
        avg_ride_minutes=("ride_length", "mean"),
        median_ride_minutes=("ride_length", "median"),
    )
    .reset_index()
    .sort_values("trips", ascending=False)
)

rides_by_day = (
    trips.groupby(["member_casual", "day_name"], dropna=False)
    .size()
    .reset_index(name="trips")
)

rides_by_month = (
    trips.groupby(["member_casual", "month_name"], dropna=False)
    .size()
    .reset_index(name="trips")
)

display(rider_mix)
display(rides_by_day.head(14))
display(rides_by_month.head(24))


,member_casual,trips,avg_ride_minutes,median_ride_minutes
1,member,3514069,11.81,8.75
0,casual,1925093,18.55,11.86


,member_casual,day_name,trips
0,casual,Friday,308397
1,casual,Monday,220900
2,casual,Saturday,396308
3,casual,Sunday,318209
4,casual,Thursday,248272
5,casual,Tuesday,218954
6,casual,Wednesday,214053
7,member,Friday,522778
8,member,Monday,496329
9,member,Saturday,441451


,member_casual,month_name,trips
0,casual,April,105057
1,casual,August,322855
2,casual,December,27027
3,casual,February,40022
4,casual,January,23800
5,casual,July,307693
6,casual,June,277983
7,casual,March,82709
8,casual,May,175230
9,casual,November,94522



## 11. Export processed outputs

The cleaned dataset and audit outputs are written to `02_Processed_Data`.


In [23]:
cleaned_csv_path = PROCESSED_DIR / "cyclistic_final_cleaned.csv"
cleaned_parquet_path = PROCESSED_DIR / "cyclistic_final_cleaned.parquet"

# 2. Define the 28 columns we want for SQL/Analysis
# We use the current dataframe columns to avoid NameErrors
final_columns = trips.columns.tolist()

print(f"Exporting {len(trips):,} rows and {len(final_columns)} columns...")

# 3. Export to CSV (For SQL, Power BI, Tableau)
trips.to_csv(cleaned_csv_path, index=False)
print(f"CSV Saved: {cleaned_csv_path}")

# 4. Export to Parquet (For your next Python notebooks)
trips.to_parquet(cleaned_parquet_path, index=False, engine='pyarrow')
print(f"Parquet Saved: {cleaned_parquet_path}")

Exporting 5,439,162 rows and 28 columns...
CSV Saved: C:\Users\John Lloyd Legaspi\Documents\Projects\Cyclistic_Bike_Share\02_Processed_Data\cyclistic_final_cleaned.csv
Parquet Saved: C:\Users\John Lloyd Legaspi\Documents\Projects\Cyclistic_Bike_Share\02_Processed_Data\cyclistic_final_cleaned.parquet
